# Run ETL

Notebook version of `Cloud/run_etl.py`. It reads raw data, engineers features, splits the dataset, and saves processed data.

In [ ]:
from pathlib import Path
import os
import sys

# Make imports work when this notebook is opened from Cloud/pipeline.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "pipeline":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if sys.platform.startswith("win"):
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
        sys.stderr.reconfigure(encoding="utf-8", errors="replace")
    except AttributeError:
        pass

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import pandas as pd

from etl.etl_pipeline import StockETLPipeline
from configs.config import Config

In [ ]:
config = Config()

# Create processed data directory if it does not exist.
os.makedirs(config.data.get("processed_data_path", "./data/processed"), exist_ok=True)

print("=" * 60)
print("STEP 2: ETL PROCESSING - Process data and create features")
print("=" * 60)

raw_data_path = os.path.join(config.data["raw_data_path"], "combined_stock_data.parquet")
print(f"\nLoad raw data from: {raw_data_path}\n")
df = pd.read_parquet(raw_data_path)

print(f"Input records: {len(df)}")
print(f"Input columns: {list(df.columns)}\n")

In [ ]:
etl_pipeline = StockETLPipeline(
    ma_windows=config.etl["moving_average_windows"],
    lag_windows=config.etl["lag_windows"],
    rsi_period=config.etl["rsi_period"],
)

print("Processing pipeline:")
print("   - Handling missing values")
print("   - Removing outliers")
print("   - Computing moving averages (MA10, MA20, MA50)")
print("   - Computing lag features (Lag1, Lag5, Lag10)")
print("   - Computing RSI indicator")
print("   - Computing daily returns and volatility")
print("   - Creating target variable")
print("   - Standardizing features")
print("   - Splitting train/val/test (70/10/20)\n")

In [ ]:
result = etl_pipeline.run_etl_pipeline(df)

processed_df = result["processed_df"]
train_data, val_data, test_data = result["train_data"], result["val_data"], result["test_data"]

print("ETL processing completed!\n")
print("Output Statistics:")
print(f"   Processed records: {len(processed_df)}")
print(f"   Features engineered: {len(processed_df.columns) - 3}")
print(f"   Train set: {len(train_data[0])} samples")
print(f"   Val set: {len(val_data[0])} samples")
print(f"   Test set: {len(test_data[0])} samples")
print(f"   Total: {len(train_data[0]) + len(val_data[0]) + len(test_data[0])}")
print(f"\nFeatures: {list(processed_df.columns)}\n")

In [ ]:
processed_path = os.path.join(
    config.data.get("processed_data_path", "./data/processed"),
    "processed_stock_data.parquet",
)
processed_df.to_parquet(processed_path, index=False)
print(f"Processed data saved: {processed_path}\n")